# HPO LunarLander

Optuna hyperparameter optimization for `VectorTrainer` on LunarLander.

The happy path on Colab is labeled HP1 to HP4.

Hardware: L4 GPU is the intended runtime.

## Setup

### Runtime setup

Set up the runtime by running exactly one code cell in this section.

#### Colab

In [ ]:
# HP1
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r hpo/requirements.txt

from pathlib import Path
import sys

from google.colab import drive

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.lunar_lander.logging import configure_file_logging

study_dir = Path("/content/drive/MyDrive/rl_lab/hpo")
drive.mount("/content/drive")
study_dir.mkdir(parents=True, exist_ok=True)
configure_file_logging(study_dir)

HPO_STUDY_DIR = study_dir

#### Local

In [ ]:
# Local setup
from pathlib import Path
import sys
sys.path.insert(0, "../../dqn/src")
sys.path.insert(0, "../src")
HPO_STUDY_DIR = Path("../runs")
HPO_STUDY_DIR.mkdir(parents=True, exist_ok=True)


## Imports

In [ ]:
# HP2
import torch

from hpo.evaluation.dashboard import Dashboard
from hpo.lunar_lander.environment import EnvFactory
from hpo.objective import ObjectiveConfig
from hpo.study import Baseline, StudyRunner
from hpo.notebook_utils import neighbors

ENVIRONMENT_FACTORY = EnvFactory()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OBJECTIVE_CFG = ObjectiveConfig(
    environment_factory=ENVIRONMENT_FACTORY,
    device=device,
)
device

## Optimization

### Define SearchSpaces

In [ ]:
# HP3
BATCH_SIZES = [256, 512, 1024]
LEARNING_STARTS = [2_500, 5_000, 10_000]
OPTIMIZE_EVERY = [2, 4, 8]
REPLAY_CAPACITIES = [50_000, 100_000, 200_000]


BASELINE_PARAMS = {
    "learning_rate": 3e-4,
    "batch_size": 512,
    "eps_end": 0.05,
    "eps_decay": 25_000,
    "gamma": 0.99,
    "tau": 0.005,
    "learning_starts": 5_000,
    "optimize_every": 4,
    "replay_memory_capacity": 100_000,
    "num_episodes": 600,
}


def suggest_s1_params(trial, incumbent_params):
    trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)
    trial.suggest_categorical("batch_size", BATCH_SIZES)
    trial.suggest_categorical("learning_starts", LEARNING_STARTS)
    trial.suggest_categorical("optimize_every", OPTIMIZE_EVERY)


def suggest_s2_params(trial, incumbent_params):
    trial.suggest_float("eps_end", 0.01, 0.10)
    trial.suggest_int("eps_decay", 10_000, 60_000, log=True)


def suggest_s3_params(trial, incumbent_params):
    trial.suggest_categorical("replay_memory_capacity", REPLAY_CAPACITIES)


def suggest_s4_params(trial, incumbent_params):
    eps_end = incumbent_params["eps_end"]
    eps_decay = incumbent_params["eps_decay"]
    trial.suggest_float(
        "learning_rate",
        incumbent_params["learning_rate"] / 2,
        incumbent_params["learning_rate"] * 2,
        log=True,
    )
    trial.suggest_categorical(
        "batch_size",
        neighbors(incumbent_params["batch_size"], BATCH_SIZES),
    )
    trial.suggest_float(
        "eps_end",
        max(0.01, eps_end - 0.02),
        min(0.10, eps_end + 0.02),
    )
    trial.suggest_int(
        "eps_decay",
        max(1, eps_decay // 2),
        eps_decay * 2,
        log=True,
    )
    trial.suggest_categorical(
        "learning_starts",
        neighbors(incumbent_params["learning_starts"], LEARNING_STARTS),
    )
    trial.suggest_categorical(
        "optimize_every",
        neighbors(incumbent_params["optimize_every"], OPTIMIZE_EVERY),
    )


### Optimize

In [ ]:
# HP4
runner = StudyRunner(
    database_path=lambda name: HPO_STUDY_DIR / f"{name}.db",
    objective_cfg=OBJECTIVE_CFG,
    baseline=Baseline(params=BASELINE_PARAMS, score=0.0),
    reporter=Dashboard(),
)

runner.run("s1_update_economy", suggest_s1_params, 40)
runner.run("s2_exploration", suggest_s2_params, 40)
runner.run("s3_replay_capacity", suggest_s3_params, 10)
runner.run("s4_joint_finetune", suggest_s4_params, 30)


## Analysis

### Load studies (only after runtime reset)

In [ ]:
# Run only after the runtime environment has been reset.
!pip install -q optuna plotly nbformat

from pathlib import Path
from google.colab import drive
import optuna

drive.mount("/content/drive")
HPO_STUDY_DIR = Path("/content/drive/MyDrive/rl_lab/hpo")

def load_study(name):
    return optuna.load_study(
        study_name=name,
        storage=f"sqlite:///{HPO_STUDY_DIR / f'{name}.db'}",
    )

study1 = load_study("s1_update_economy")
study2 = load_study("s2_exploration")
study3 = load_study("s3_replay_capacity")
study4 = load_study("s4_joint_finetune")


### Best hyperparameters

In [ ]:
import pandas as pd

studies = [study1, study2, study3, study4]
columns = ["S1 Update economy", "S2 Exploration", "S3 Replay capacity", "S4 Joint fine-tune"]
table = pd.DataFrame({
    column: study.user_attrs["incumbent_params"]
    for column, study in zip(columns, studies)
})
table.loc["score"] = [study.user_attrs["incumbent_score"] for study in studies]

integer_rows = ["batch_size", "eps_decay", "learning_starts", "optimize_every", "replay_memory_capacity"]
styled_table = table.style.format("{:.3f}")
styled_table = styled_table.format("{:.6f}", subset=pd.IndexSlice[["learning_rate"], :])
styled_table = styled_table.format("{:.0f}", subset=pd.IndexSlice[integer_rows, :])
styled_table = styled_table.format("{:.3f}", subset=pd.IndexSlice[["score"], :])
styled_table = styled_table.set_table_styles([{
    "selector": "tbody tr:last-child th",
    "props": "border-top: 2px solid #888; font-weight: bold",
}])
display(styled_table)


### Plots

Score dependence on the hyperparameters optimized in each study.

In [ ]:
from IPython.display import display
from optuna.visualization import plot_contour, plot_slice

print("S1: Update economy")
display(plot_contour(study1, target_name="Score"))
print("S2: Exploration")
display(plot_contour(study2, target_name="Score"))
print("S3: Replay capacity")
display(plot_slice(study3, target_name="Score"))
print("S4: Joint fine-tune")
display(plot_contour(study4, target_name="Score"))
